# model_tabnet.py

TabNet (Deep Learning) for Heart Disease / Cardiovascular Risk prediction.

TabNet – Attentive Interpretable Tabular Learning (Arik & Pfister, 2021)
Library: pytorch-tabnet  (pip install pytorch-tabnet)

Steps:
  1. Load preprocessed data
  2. Define model architecture
  3. Loss: cross-entropy (TabNetClassifier default)
  4. Optimiser: Adam with LR scheduler
  5. Train for N epochs in batches with early stopping
  6. Evaluate
  7. Fine-tune (wider hyperparameter search)
  8. Save model
  9. Predict on new data

Install dependency:
    pip install pytorch-tabnet

In [1]:
pip install pytorch-tabnet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ── Install check ──────────────────────────────────────────────────────────────
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    from pytorch_tabnet.augmentations import ClassificationSMOTE
    TABNET_AVAILABLE = True
except ImportError:
    TABNET_AVAILABLE = False
    print("="*70)
    print("pytorch-tabnet is NOT installed.")
    print("Run:  pip install pytorch-tabnet")
    print("Then re-run this script.")
    print("="*70)

import numpy as np
import pandas as pd
import joblib, os
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

if not TABNET_AVAILABLE:
    raise SystemExit("Please install pytorch-tabnet first.")

# ── Paths ──────────────────────────────────────────────────────────────────────
PRE  = "preprocessed"
MDIR = "models/tabnet"; os.makedirs(MDIR, exist_ok=True)
PDIR = "plots/tabnet";  os.makedirs(PDIR, exist_ok=True)
LABEL = {0: 'Low', 1: 'Moderate', 2: 'High'}

## 1. Load preprocessed splits

In [3]:
X_train = pd.read_csv(f"{PRE}/X_train.csv").values.astype(np.float32)
X_val   = pd.read_csv(f"{PRE}/X_val.csv").values.astype(np.float32)
X_test  = pd.read_csv(f"{PRE}/X_test.csv").values.astype(np.float32)
y_train = pd.read_csv(f"{PRE}/y_train.csv").squeeze().values.astype(int)
y_val   = pd.read_csv(f"{PRE}/y_val.csv").squeeze().values.astype(int)
y_test  = pd.read_csv(f"{PRE}/y_test.csv").squeeze().values.astype(int)
class_weights = joblib.load(f"{PRE}/class_weights.pkl")
feature_names = joblib.load(f"{PRE}/feature_names.pkl")
print(f"Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")

Train (51229, 30) | Val (10978, 30) | Test (10978, 30)


## 2. Define model architecture N_D = N_A = width of decision/attention embeddings N_STEPS = number of sequential attention steps GAMMA   = coefficient for feature reusage in attention

In [4]:
# 3. Loss: cross-entropy (built into TabNetClassifier)
# 4. Optimiser: Adam, with LR scheduling

tabnet = TabNetClassifier(
    n_d             = 32,           # Decision step width
    n_a             = 32,           # Attention embedding width
    n_steps         = 4,            # Sequential attention steps
    gamma           = 1.3,          # Feature reuse penalty
    n_independent   = 2,            # GLU layers per step (independent)
    n_shared        = 2,            # Shared GLU layers
    lambda_sparse   = 1e-3,         # Sparsity regularisation (promotes feature selection)
    optimizer_fn    = __import__('torch').optim.Adam,
    optimizer_params= {'lr': 2e-3, 'weight_decay': 1e-5},   # 4. LR
    scheduler_fn    = __import__('torch').optim.lr_scheduler.StepLR,
    scheduler_params= {'step_size': 10, 'gamma': 0.9},      # LR decay
    mask_type       = 'sparsemax',  # attention mask (sparsemax | entmax)
    input_dim       = X_train.shape[1],
    output_dim      = 3,
    verbose         = 10,
    seed            = 42,
)

## 5. Train for N epochs in batches with early stopping

In [5]:
N_EPOCHS    = 100
BATCH_SIZE  = 512
VBATCH_SIZE = 256
PATIENCE    = 15           # early stopping patience

# Class weights array for TabNet
weights = np.array([class_weights[k] for k in sorted(class_weights)])

print("\n── Training TabNet ──")
tabnet.fit(
    X_train            = X_train,
    y_train            = y_train,
    eval_set           = [(X_train, y_train), (X_val, y_val)],
    eval_name          = ['train', 'val'],
    eval_metric        = ['accuracy', 'logloss'],
    max_epochs         = N_EPOCHS,
    patience           = PATIENCE,
    batch_size         = BATCH_SIZE,
    virtual_batch_size = VBATCH_SIZE,
    weights            = 1,          # use class weights from class_weight param
    drop_last          = False,
    augmentations      = ClassificationSMOTE(p=0.2),   # on-the-fly augmentation
)

print(f"\nBest epoch : {tabnet.best_epoch}")


── Training TabNet ──
epoch 0  | loss: 1.09148 | train_accuracy: 0.6567  | train_logloss: 0.86005 | val_accuracy: 0.66069 | val_logloss: 0.85786 |  0:00:06s
epoch 10 | loss: 0.45385 | train_accuracy: 0.8903  | train_logloss: 0.32041 | val_accuracy: 0.88213 | val_logloss: 0.33344 |  0:01:10s
epoch 20 | loss: 0.35117 | train_accuracy: 0.92643 | train_logloss: 0.20707 | val_accuracy: 0.91647 | val_logloss: 0.23819 |  0:02:31s
epoch 30 | loss: 0.28075 | train_accuracy: 0.93771 | train_logloss: 0.17274 | val_accuracy: 0.92111 | val_logloss: 0.22243 |  0:03:50s
epoch 40 | loss: 0.23895 | train_accuracy: 0.95167 | train_logloss: 0.13391 | val_accuracy: 0.9325  | val_logloss: 0.20132 |  0:05:10s
epoch 50 | loss: 0.21689 | train_accuracy: 0.96354 | train_logloss: 0.10548 | val_accuracy: 0.94562 | val_logloss: 0.17182 |  0:06:28s
epoch 60 | loss: 0.19682 | train_accuracy: 0.96963 | train_logloss: 0.08965 | val_accuracy: 0.94917 | val_logloss: 0.16039 |  0:07:45s
epoch 70 | loss: 0.17965 | train

## 6. Evaluate on Test Set

In [6]:
print("\n── Evaluation on Test Set ──")
y_pred  = tabnet.predict(X_test)
y_proba = tabnet.predict_proba(X_test)
print(classification_report(y_test, y_pred, target_names=[LABEL[i] for i in sorted(LABEL)]))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba, multi_class='ovr'))

# Feature importance from TabNet (mean attention masks)
feat_imp = pd.Series(tabnet.feature_importances_, index=feature_names)
print("\nTop 10 important features:\n", feat_imp.nlargest(10))


── Evaluation on Test Set ──
              precision    recall  f1-score   support

         Low       0.99      0.97      0.98      7669
    Moderate       0.97      0.96      0.96      3173
        High       0.12      0.30      0.17       136

    accuracy                           0.96     10978
   macro avg       0.69      0.74      0.71     10978
weighted avg       0.97      0.96      0.97     10978

Accuracy : 0.9583712880306067
ROC-AUC  : 0.905459963170987

Top 10 important features:
 Risk_Composite        0.152285
Smoking_Alcohol       0.090245
Diabetes              0.089585
Smoking_History       0.084924
Depression            0.064157
Sex_Female            0.058234
Weight_(kg)           0.051173
Alcohol_per_Weight    0.049545
General_Health        0.036521
Height_to_Weight      0.035263
dtype: float64


## 7. Fine-Tune (re-train with adjusted hyperparams)

In [7]:
print("\n── Fine-Tuning TabNet ──")
tabnet_ft = TabNetClassifier(
    n_d             = 64,
    n_a             = 64,
    n_steps         = 5,
    gamma           = 1.5,
    n_independent   = 2,
    n_shared        = 3,
    lambda_sparse   = 5e-4,
    optimizer_fn    = __import__('torch').optim.Adam,
    optimizer_params= {'lr': 1e-3, 'weight_decay': 1e-5},
    scheduler_fn    = __import__('torch').optim.lr_scheduler.CosineAnnealingLR,
    scheduler_params= {'T_max': 50, 'eta_min': 1e-5},
    mask_type       = 'entmax',
    input_dim       = X_train.shape[1],
    output_dim      = 3,
    verbose         = 10,
    seed            = 42,
)

# Combine train + val for fine-tuning
X_tv = np.vstack([X_train, X_val])
y_tv = np.concatenate([y_train, y_val])

tabnet_ft.fit(
    X_train            = X_tv,
    y_train            = y_tv,
    eval_set           = [(X_test, y_test)],
    eval_name          = ['test'],
    eval_metric        = ['accuracy', 'logloss'],
    max_epochs         = 150,
    patience           = 20,
    batch_size         = 512,
    virtual_batch_size = 256,
    weights            = 1,
    drop_last          = False,
)

y_pred_ft  = tabnet_ft.predict(X_test)
y_proba_ft = tabnet_ft.predict_proba(X_test)
print("Fine-tuned Accuracy :", accuracy_score(y_test, y_pred_ft))
print("Fine-tuned ROC-AUC  :", roc_auc_score(y_test, y_proba_ft, multi_class='ovr'))
print(classification_report(y_test, y_pred_ft, target_names=[LABEL[i] for i in sorted(LABEL)]))


── Fine-Tuning TabNet ──
epoch 0  | loss: 1.2365  | test_accuracy: 0.70796 | test_logloss: 0.82878 |  0:00:08s
epoch 10 | loss: 0.29513 | test_accuracy: 0.92275 | test_logloss: 0.22939 |  0:01:33s
epoch 20 | loss: 0.15836 | test_accuracy: 0.94006 | test_logloss: 0.18807 |  0:02:48s
epoch 30 | loss: 0.10425 | test_accuracy: 0.95263 | test_logloss: 0.17197 |  0:04:09s
epoch 40 | loss: 0.07567 | test_accuracy: 0.96183 | test_logloss: 0.15678 |  0:05:36s
epoch 50 | loss: 0.07311 | test_accuracy: 0.96375 | test_logloss: 0.15747 |  0:07:00s
epoch 60 | loss: 0.07258 | test_accuracy: 0.96347 | test_logloss: 0.16184 |  0:08:27s

Early stopping occurred at epoch 63 with best_epoch = 43 and best_test_logloss = 0.15368
Fine-tuned Accuracy : 0.9645654946256149
Fine-tuned ROC-AUC  : 0.891842212314684
              precision    recall  f1-score   support

         Low       0.99      0.98      0.98      7669
    Moderate       0.98      0.96      0.97      3173
        High       0.15      0.33     

## 8. Save models

In [8]:
tabnet.save_model(f"{MDIR}/tabnet_base")          # saves .zip
tabnet_ft.save_model(f"{MDIR}/tabnet_finetuned")
print(f"✅ Models saved to ./{MDIR}/")

Successfully saved model at models/tabnet/tabnet_base.zip
Successfully saved model at models/tabnet/tabnet_finetuned.zip
✅ Models saved to ./models/tabnet/


## Plots

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Loss curve
history = tabnet.history
axes[0].plot(history['train_logloss'], label='Train logloss')
axes[0].plot(history['val_logloss'],   label='Val logloss')
axes[0].set(title='TabNet Training Curve', xlabel='Epoch', ylabel='Logloss'); axes[0].legend()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_ft)
ConfusionMatrixDisplay(cm, display_labels=[LABEL[i] for i in sorted(LABEL)]).plot(ax=axes[1])
axes[1].set_title('TabNet Confusion Matrix (fine-tuned)')

# Feature importance
feat_imp_ft = pd.Series(tabnet_ft.feature_importances_, index=feature_names).nlargest(20)
feat_imp_ft.sort_values().plot(kind='barh', ax=axes[2])
axes[2].set_title('Top-20 TabNet Feature Importances')

plt.tight_layout()
plt.savefig(f"{PDIR}/tabnet_results.png", dpi=150)
print(f"Plot saved to ./{PDIR}/tabnet_results.png")

Plot saved to ./plots/tabnet/tabnet_results.png


## 9. Predict on new data

In [10]:
def predict_new(raw_df: pd.DataFrame, model_zip: str = f"{MDIR}/tabnet_finetuned.zip"):
    """
    Accepts a pre-processed DataFrame (post preprocessing.py).
    Returns (predicted_labels, probabilities).
    """
    from pytorch_tabnet.tab_model import TabNetClassifier
    scaler        = joblib.load(f"{PRE}/scaler.pkl")
    feature_names = joblib.load(f"{PRE}/feature_names.pkl")

    model = TabNetClassifier()
    model.load_model(model_zip)

    raw_df = raw_df.reindex(columns=feature_names, fill_value=0)

    numerical_cols = [
        'Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption',
        'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption',
        'Checkup_Frequency', 'Lifestyle_Score', 'Healthy_Diet_Score',
        'Smoking_Alcohol', 'Checkup_Exercise', 'Height_to_Weight',
        'Fruit_Vegetables', 'HealthyDiet_Lifestyle', 'Alcohol_FriedPotato',
        'Risk_Composite', 'Alcohol_per_Weight'
    ]
    num_present = [c for c in numerical_cols if c in raw_df.columns]
    raw_df[num_present] = scaler.transform(raw_df[num_present])

    arr    = raw_df.values.astype(np.float32)
    preds  = model.predict(arr)
    probas = model.predict_proba(arr)
    return [LABEL[p] for p in preds], probas

# Demo (using base model on test set)
demo_preds = tabnet.predict(X_test[:5])
print("\nDemo predictions:", [LABEL[p] for p in demo_preds],
      "| Actual:", [LABEL[p] for p in y_test[:5]])


Demo predictions: ['High', 'Moderate', 'Low', 'Low', 'Low'] | Actual: ['Low', 'Moderate', 'Low', 'Low', 'Low']
